<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


<h1>Lab: Exploring and Loading a Text Corpus with PyTorch</h1>


Estimated time needed: **40** minutes


## Learning Objectives

After completing this lab, you will be able to:

 - Load and explore a text corpus using PyTorch utilities and datasets, and analyze likely preprocessing challenges with AI-assisted support
 - Implement tokenization and build a vocabulary from raw text data
 - Convert text data into numerical representations
 - Apply padding and truncation to manage variable-length sequences
 - Create a PyTorch DataLoader for batching and efficient training


## __Table of Contents__

<p>In this lab, you will explore a text corpus and prepare it for sequence modeling with PyTorch.</p>

<ul>
    <li><a href="#Create-and-Explore-a-Text-Corpus">Create and Explore a Text Corpus</a></li>
    <li><a href="#Analyze-Preprocessing-Challenges">Analyze Preprocessing Challenges</a></li>
    <li><a href="#Tokenization">Tokenization</a></li>
    <li><a href="#Vocabulary-Mapping">Vocabulary Mapping</a></li>
    <li><a href="#Convert-Text-to-Token-IDs">Convert Text to Token IDs</a></li>
    <li><a href="#Padding-and-Truncation">Padding and Truncation</a></li>
    <li><a href="#Create-a-PyTorch-DataLoader">Create a PyTorch DataLoader</a></li>
   
</ul>

<hr>


----


### Installing and Importing Required Libraries


The following required libraries are __not__ pre-installed in the Skills Network Labs environment. __You will need to run the following cells__ to install or import them:


In [1]:
# Install PyTorch

!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 722.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Import the libraries we need for this lab
import re
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(2)

/opt/conda/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))

## Create and Explore a Text Corpus

A text corpus is a collection of text examples.

In this lab, each example also has a label so it can be used later for a supervised learning task:


In [3]:
# Create a small text corpus:

corpus = [
    ("PyTorch makes building neural networks flexible and clear.", 1),
    ("The model failed because the training text was noisy.", 0),
    ("Tokenization splits raw text into smaller useful units.", 1),
    ("Short documents and very long documents need careful batching.", 0),
    ("Vocabulary mapping converts words into integer token IDs.", 1),
    ("Padding helps DataLoader create equal length tensors.", 1),
]

Print the number of examples and preview the first text sample:


In [4]:
# Explore the corpus

print("Number of examples:", len(corpus))
print("First example:", corpus[0])


Number of examples: 6
First example: ('PyTorch makes building neural networks flexible and clear.', 1)

Separate text examples from their labels:


In [5]:
# Split text and labels

texts = [item[0] for item in corpus]
labels = [item[1] for item in corpus]

print(texts[:2])
print(labels[:2])


['PyTorch makes building neural networks flexible and clear.', 'The model failed because the training text was noisy.']
[1, 0]

## Analyze Preprocessing Challenges

Before training a model, review the corpus for likely preprocessing issues such as punctuation, mixed casing, rare words, and variable sequence lengths. AI-assisted support can help summarize these issues, but the final preprocessing decisions should be checked against the data and model goal.


In [6]:
# Analyze likely preprocessing challenges

def inspect_text_challenges(text_examples):
    lengths = [len(text.split()) for text in text_examples]
    has_uppercase = any(any(char.isupper() for char in text) for text in text_examples)
    has_punctuation = any(bool(re.search(r"[^\w\s]", text)) for text in text_examples)
    return {
        "min_words": min(lengths),
        "max_words": max(lengths),
        "has_uppercase": has_uppercase,
        "has_punctuation": has_punctuation,
    }

inspect_text_challenges(texts)


{'min_words': 7,
 'max_words': 9,
 'has_uppercase': True,
 'has_punctuation': True}

The output helps identify preprocessing choices, such as lowercasing, punctuation handling, and padding or truncation length.


## Tokenization

Tokenization converts raw text into smaller pieces called tokens. Here, use a simple word tokenizer that lowercases the text and keeps only word characters:


In [7]:
# Create a tokenizer

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


Apply the tokenizer to each text example:


In [8]:
# Tokenize the corpus

tokenized_texts = [tokenize(text) for text in texts]

for tokens in tokenized_texts[:3]:
    print(tokens)


['pytorch', 'makes', 'building', 'neural', 'networks', 'flexible', 'and', 'clear']
['the', 'model', 'failed', 'because', 'the', 'training', 'text', 'was', 'noisy']
['tokenization', 'splits', 'raw', 'text', 'into', 'smaller', 'useful', 'units']

## Vocabulary Mapping

A vocabulary maps each token to an integer ID. Reserve special IDs for padding and unknown tokens:


In [9]:
# Build a vocabulary from tokens

special_tokens = ["<pad>", "<unk>"]
token_counts = Counter(token for tokens in tokenized_texts for token in tokens)

vocab = {token: index for index, token in enumerate(special_tokens)}

for token, count in token_counts.items():
    if token not in vocab:
        vocab[token] = len(vocab)

print("Vocabulary size:", len(vocab))
print(list(vocab.items())[:10])


Vocabulary size: 46
[('<pad>', 0), ('<unk>', 1), ('pytorch', 2), ('makes', 3), ('building', 4), ('neural', 5), ('networks', 6), ('flexible', 7), ('and', 8), ('clear', 9)]

Use the reserved token IDs for padding and unknown words:


In [10]:
# Store special token IDs

pad_id = vocab["<pad>"]
unk_id = vocab["<unk>"]

print("PAD ID:", pad_id)
print("UNK ID:", unk_id)


PAD ID: 0
UNK ID: 1

## Convert Text to Token IDs

Convert each token into its vocabulary ID. If a token is not in the vocabulary, use the unknown token ID:


In [11]:
# Convert tokens to token IDs

def tokens_to_ids(tokens, vocabulary):
    return [vocabulary.get(token, vocabulary["<unk>"]) for token in tokens]

encoded_texts = [tokens_to_ids(tokens, vocab) for tokens in tokenized_texts]

print(tokenized_texts[0])
print(encoded_texts[0])


['pytorch', 'makes', 'building', 'neural', 'networks', 'flexible', 'and', 'clear']
[2, 3, 4, 5, 6, 7, 8, 9]

Test how the vocabulary handles a word that was not seen in the original corpus:


In [12]:
# Encode text with an unknown word

new_text = "PyTorch supports embeddings"
new_tokens = tokenize(new_text)
new_ids = tokens_to_ids(new_tokens, vocab)

print(new_tokens)
print(new_ids)


['pytorch', 'supports', 'embeddings']
[2, 1, 1]

## Padding and Truncation

Neural networks usually process batches of equal-length tensors. Padding adds tokens to short sequences, and truncation shortens sequences that are too long:


In [13]:
# Pad or truncate a sequence

def pad_or_truncate(sequence, max_length, padding_id):
    if len(sequence) > max_length:
        return sequence[:max_length]
    return sequence + [padding_id] * (max_length - len(sequence))


Apply the function to every encoded sequence:


In [14]:
# Create fixed-length sequences

max_length = 8
padded_texts = [pad_or_truncate(sequence, max_length, pad_id) for sequence in encoded_texts]

print("Original length:", len(encoded_texts[0]))
print("Padded length:", len(padded_texts[0]))
print(padded_texts[0])


Original length: 8
Padded length: 8
[2, 3, 4, 5, 6, 7, 8, 9]

Convert the padded examples and labels to PyTorch tensors:


In [15]:
# Create tensors

features = torch.tensor(padded_texts, dtype=torch.long)
targets = torch.tensor(labels, dtype=torch.long)

print(features.shape)
print(targets.shape)


torch.Size([6, 8])
torch.Size([6])

## Create a PyTorch DataLoader

A PyTorch Dataset stores examples, and a DataLoader creates batches for efficient model training:


In [16]:
# Create a PyTorch Dataset

class TextCorpusDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


dataset = TextCorpusDataset(features, targets)

print("Dataset size:", len(dataset))
print(dataset[0])


Dataset size: 6
(tensor([2, 3, 4, 5, 6, 7, 8, 9]), tensor(1))

Create a DataLoader and inspect one batch:


In [17]:
# Create a DataLoader for batching

loader = DataLoader(dataset, batch_size=2, shuffle=True)

for batch_features, batch_labels in loader:
    print("Batch features shape:", batch_features.shape)
    print("Batch labels shape:", batch_labels.shape)
    print(batch_features)
    print(batch_labels)
    break


Batch features shape: torch.Size([2, 8])
Batch labels shape: torch.Size([2])
tensor([[32, 33, 34, 35, 21, 36, 37, 38],
        [25, 26,  8, 27, 28, 26, 29, 30]])
tensor([1, 0])

<a href="https://dataplatform.cloud.ibm.com/registration/stepone?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork&context=cpdaas&apps=data_science_experience%2Cwatson_machine_learning"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork/Template/module%201/images/Watson_Studio.png"></a>


## Author/s


<b>Harish Pant</b> is an accomplished IT Consultant specializing in bridging the gap between complex AI technologies and practical business applications. Holding a Bachelor’s degree in Computer Science with a strong background in Machine Learning, Harish serves clients across diverse industries—including fintech, retail, and manufacturing—helping them navigate AI adoption. Harish is known for translating technical AI concepts into actionable business strategies and delivering engaging, high-impact presentations. 


Copyright © IBM Corporation. All rights reserved.
